In [ ]:
%pip install -U langgraph


  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.19
    Uninstalling langgraph-1.2.19:
      Successfully uninstalled langgraph-1.2.19
Note: you may need to restart the kernel to use updated packages.


In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv
import re

In [2]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

api_key = getpass("Cole sua OpenAI API Key: ").strip()

os.environ["OPENAI_API_KEY"] = api_key

llm = ChatOpenAI(
    model="gpt-5.4-mini",
    api_key=api_key
)

In [3]:
class State(TypedDict):
    """ Representa o estado do processo de avaliação da redação """
    essay: str
    relevance_score: float
    grammar_score: float
    structure_score: float
    depth_score: float
    final_score: float

In [4]:
#Funções de Avaliação
#Esta célula define as funções utilizadas no processo de avaliação, incluindo a extração de pontuações e os componentes individuais de avaliação.

In [5]:
def extract_score(content: str) -> float:
    """Extrai a pontuação numérica da resposta do LLM."""
    match = re.search(r'Pontuação:\s*(\d+(\.\d+)?)', content)
    if match:
        return float(match.group(1))
    raise ValueError(f"Não foi possível extrair a pontuação de: {content}")

def check_relevance(state: State) -> State:
    """Verifica a relevância da redação."""
    prompt = ChatPromptTemplate.from_template(
        "Analise a relevância da seguinte redação em relação ao tema dado, prezando pela excelência da Língua Portuguesa. "
        "Forneça uma pontuação de relevância entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nRedação: {essay}"
    )
    result = llm.invoke(prompt.format(essay=state["essay"]))
    try:
        state["relevance_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em check_relevance: {e}")
        state["relevance_score"] = 0.0
    return state

def check_grammar(state: State) -> State:
    """Verifica a gramática da redação."""
    prompt = ChatPromptTemplate.from_template(
        "Analise a gramática da Língua Portuguesa na seguinte redação. "
        "Forneça uma pontuação de gramática entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nRedação: {essay}"
    )
    result = llm.invoke(prompt.format(essay=state["essay"]))
    try:
        state["grammar_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em check_grammar: {e}")
        state["grammar_score"] = 0.0
    return state

def analyze_structure(state: State) -> State:
    """Analisa a estrutura da redação."""
    prompt = ChatPromptTemplate.from_template(
        "Analise a estrutura de acordo com a normal culta da Língua Portuguesa na seguinte redação. "
        "Forneça uma pontuação de estrutura entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nRedação: {essay}"
    )
    result = llm.invoke(prompt.format(essay=state["essay"]))
    try:
        state["structure_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em analyze_structure: {e}")
        state["structure_score"] = 0.0
    return state

def evaluate_depth(state: State) -> State:
    """Avalia a profundidade de análise na redação."""
    prompt = ChatPromptTemplate.from_template(
        "Avalie a profundidade de análise na seguinte redação. "
        "Forneça uma pontuação de profundidade entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nRedação: {essay}"
    )
    result = llm.invoke(prompt.format(essay=state["essay"]))
    try:
        state["depth_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em evaluate_depth: {e}")
        state["depth_score"] = 0.0
    return state

def calculate_final_score(state: State) -> State:
    """Calcula a pontuação final com base nas pontuações dos componentes individuais."""
    state["final_score"] = (
        state["relevance_score"] * 0.3 +
        state["grammar_score"] * 0.2 +
        state["structure_score"] * 0.2 +
        state["depth_score"] * 0.3
    )
    return state

In [6]:
#Definição do fluxo de trabalho
#Essa célula define o fluxo de trabalho de avaliação usando StateGraph.

In [7]:
# Inicializa o StateGraph
workflow = StateGraph(State)

# Adiciona nós ao grafo
workflow.add_node("check_relevance", check_relevance)
workflow.add_node("check_grammar", check_grammar)
workflow.add_node("analyze_structure", analyze_structure)
workflow.add_node("evaluate_depth", evaluate_depth)
workflow.add_node("calculate_final_score", calculate_final_score)

# Define as funções de decisão

# def decidir_relevancia(state):
#     if state["relevance_score"] > 0.5:
#         return "check_grammar"
#     else:
#         return "calculate_final_score"


# def decidir_gramatica(state):
#     if state["grammar_score"] > 0.6:
#         return "analyze_structure"
#     else:
#         return "calculate_final_score"


# def decidir_estrutura(state):
#     if state["structure_score"] > 0.7:
#         return "evaluate_depth"
#     else:
#         return "calculate_final_score"


# # Adiciona as arestas condicionais

# workflow.add_conditional_edges(
#     "check_relevance",
#     decidir_relevancia
# )

# workflow.add_conditional_edges(
#     "check_grammar",
#     decidir_gramatica
# )

# workflow.add_conditional_edges(
#     "analyze_structure",
#     decidir_estrutura
# )

# # Aresta sem condição
# workflow.add_edge(
#     "evaluate_depth",
#     "calculate_final_score"
# )




# Define e adiciona arestas condicionais
workflow.add_conditional_edges(
    "check_relevance",
    lambda x: (
        "check_grammar"
        if x["relevance_score"] > 0.5
        else "calculate_final_score"
    )
)

workflow.add_conditional_edges(
    "check_grammar",
    lambda x: (
        "analyze_structure"
        if x["relevance_score"] > 0.6
        else "calculate_final_score"
    )
)

workflow.add_conditional_edges(
    "analyze_structure",
    lambda x: (
        "evaluate_depth"
        if x["relevance_score"] > 0.7
        else "calculate_final_score"
    )
)

workflow.add_conditional_edges(
    "evaluate_depth",
    lambda x: (
    "calculate_final_score"
    )
)


    
# # Define e adiciona arestas condicionais
# workflow.add_conditional_edges(
#     "check_relevance",
#     lambda x: "check_grammar" if x["relevance_score"] > 0.5 else "calculate_final_score"
    
# )
# workflow.add_conditional_edges(
#     "check_grammar",
#     lambda x: "analyze_structure" if x["grammar_score"] > 0.6 else "calculate_final_score"
# )
# workflow.add_conditional_edges(
#     "analyze_structure",
#     lambda x: "evaluate_depth" if x["structure_score"] > 0.7 else "calculate_final_score"
# )
# workflow.add_conditional_edges(
#     "evaluate_depth",
#     lambda x: "calculate_final_score"
# )

# Define o ponto de entrada
workflow.set_entry_point("check_relevance")

# Define o ponto de saída
workflow.add_edge("calculate_final_score", END)


# Compila o grafo
app = workflow.compile()

In [8]:
# workflow.add_conditional_edges(
#     "check_relevance",
#     decidir_proximo_no
# )
# workflow.add_conditional_edges(
#     "check_grammar",
#     decidir_proximo_no
# )
# workflow.add_conditional_edges(
#     "check_relevance",
#     decidir_proximo_no
# )
# # Define e adiciona arestas condicionais
# workflow.add_conditional_edges(
#     "check_relevance",
#     lambda x: "check_grammar" if x["relevance_score"] > 0.5 else "calculate_final_score"
    
# )
# workflow.add_conditional_edges(
#     "check_grammar",
#     lambda x: "analyze_structure" if x["grammar_score"] > 0.6 else "calculate_final_score"
# )
# workflow.add_conditional_edges(
#     "analyze_structure",
#     lambda x: "evaluate_depth" if x["structure_score"] > 0.7 else "calculate_final_score"
# )
# workflow.add_conditional_edges(
#     "evaluate_depth",
#     lambda x: "calculate_final_score"
# )

# #Função de avaliação de redação
# #Essa célula define a função principal para avaliar uma redação usando o fluxo de trabalho definido.

In [9]:
def grade_essay(essay: str) -> dict:
    """Avalia a redação fornecida usando o fluxo de trabalho definido."""
    initial_state = State(
        essay=essay,
        relevance_score=0.0,
        grammar_score=0.0,
        structure_score=0.0,
        depth_score=0.0,
        final_score=0.0
    )
    result = app.invoke(initial_state)
    return result

In [10]:
sample_essay = """
O Impacto da Computação Quântica na Inteligência Artificial: Potencialidades, Desafios e Perspectivas Futuras

A inteligência artificial (IA) tem revolucionado a maneira como a sociedade processa informações, automatiza tarefas e desenvolve soluções para problemas complexos. Entretanto, à medida que os modelos de aprendizado de máquina se tornam mais sofisticados, aumentam também as demandas por recursos computacionais, memória e energia. Nesse contexto, a computação quântica surge como um paradigma alternativo de processamento, fundamentado nos princípios da mecânica quântica, com potencial para acelerar determinadas classes de problemas matemáticos e computacionais. A integração dessas tecnologias, conhecida como aprendizado de máquina quântico (*Quantum Machine Learning* — QML), representa uma área de pesquisa promissora, embora ainda existam limitações teóricas e experimentais que dificultam sua aplicação em larga escala.

Diferentemente da computação clássica, que utiliza bits para representar informações por meio dos estados 0 ou 1, a computação quântica emprega qubits, capazes de existir em superposições de estados. Além disso, fenômenos como o emaranhamento e a interferência quântica permitem construir operações computacionais que não possuem equivalentes diretos nos circuitos clássicos convencionais. Essas propriedades possibilitam que determinados algoritmos explorem estruturas matemáticas de maneira mais eficiente. Entretanto, a superposição não significa que todas as soluções de um problema possam ser obtidas simultaneamente, pois a medição dos estados quânticos impõe restrições à informação recuperável. Assim, a vantagem computacional depende da construção de algoritmos capazes de explorar a interferência quântica para aumentar a probabilidade de obtenção dos resultados desejados.

Uma das principais possibilidades de integração entre inteligência artificial e computação quântica encontra-se no desenvolvimento de novos algoritmos de aprendizado de máquina. Técnicas como máquinas de vetores de suporte com kernels quânticos, classificadores quânticos variacionais e redes neurais quânticas buscam explorar espaços de características e transformações produzidos por circuitos quânticos. Em determinados contextos, esses métodos podem representar relações complexas entre variáveis e oferecer alternativas aos algoritmos clássicos. Contudo, ainda não existe evidência de uma vantagem quântica generalizada para tarefas convencionais de classificação, regressão e reconhecimento de padrões. Além disso, a necessidade de codificar dados clássicos em estados quânticos pode introduzir custos que reduzem ou eliminam eventuais ganhos computacionais.

Outro campo relevante envolve os problemas de otimização, fundamentais para diversas aplicações de inteligência artificial. O treinamento de modelos frequentemente exige a minimização de funções de perda em espaços de parâmetros de elevada dimensão. Nesse contexto, algoritmos híbridos quântico-clássicos, como o *Quantum Approximate Optimization Algorithm* (QAOA), investigam maneiras de utilizar circuitos quânticos parametrizados para resolver determinadas formulações de otimização combinatória. Entretanto, o QAOA não representa uma solução universal para o treinamento de redes neurais, e sua vantagem sobre os métodos clássicos ainda depende das características de cada problema. Paralelamente, os chamados algoritmos quânticos variacionais utilizam processadores quânticos para calcular determinadas quantidades, enquanto computadores clássicos ajustam os parâmetros dos circuitos. Essa colaboração estabelece uma arquitetura híbrida que poderá desempenhar um papel importante no desenvolvimento da computação quântica aplicada.

A relação entre computação quântica e inteligência artificial também apresenta perspectivas importantes na pesquisa científica. Na química computacional, por exemplo, computadores quânticos poderão contribuir para simular sistemas moleculares e propriedades eletrônicas cuja representação clássica exata apresenta elevado custo computacional. Os resultados dessas simulações poderão alimentar modelos de inteligência artificial utilizados na descoberta de medicamentos, no desenvolvimento de catalisadores e na identificação de novos materiais. Dessa maneira, a computação quântica não necessariamente substituiria os algoritmos de aprendizado de máquina, mas poderia fornecer informações científicas de elevada qualidade para aprimorar sua capacidade de previsão. Essa possibilidade demonstra que a integração entre as tecnologias pode ocorrer tanto pela aceleração de algoritmos quanto pela produção de novos dados e modelos físicos.

Apesar dessas perspectivas, existem obstáculos significativos para a implementação prática dessas soluções. Os computadores quânticos atuais apresentam limitações relacionadas à quantidade e à qualidade dos qubits, à decoerência, aos erros nas operações lógicas e à necessidade de mecanismos de correção de erros. Essas dificuldades restringem a profundidade dos circuitos que podem ser executados com confiabilidade. Além disso, o treinamento de modelos quânticos variacionais pode apresentar regiões em que os gradientes da função objetivo se tornam extremamente pequenos, fenômeno conhecido como *barren plateaus*. Esse comportamento dificulta a otimização dos parâmetros e pode comprometer a escalabilidade dos algoritmos. Portanto, o avanço da inteligência artificial quântica exige não apenas melhores processadores, mas também técnicas eficientes de treinamento, estratégias de mitigação de erros e análises rigorosas de complexidade computacional.

Outra questão fundamental é a comparação entre métodos quânticos e clássicos. Embora determinados algoritmos quânticos apresentem vantagens teóricas em condições específicas, tais resultados não garantem superioridade em aplicações reais. A avaliação de desempenho precisa considerar o tempo necessário para preparar os dados, executar os circuitos, realizar medições e interpretar os resultados. Também é necessário comparar os métodos quânticos com algoritmos clássicos modernos, incluindo técnicas de computação paralela, processamento em GPUs e métodos especializados de otimização. Nesse sentido, demonstrar uma vantagem quântica útil requer comprovar benefícios mensuráveis em problemas relevantes, considerando tanto a precisão dos resultados quanto o custo computacional total.

Do ponto de vista econômico e social, a convergência entre essas tecnologias poderá modificar a dinâmica de inovação científica e industrial. Países e organizações que investirem em pesquisa quântica, desenvolvimento de hardware e formação de profissionais especializados poderão adquirir vantagens estratégicas em setores como indústria farmacêutica, segurança da informação, logística e ciência dos materiais. Entretanto, o elevado custo de implementação e a concentração de infraestrutura tecnológica levantam preocupações sobre desigualdade no acesso a essas ferramentas. Nesse cenário, políticas de incentivo à pesquisa, cooperação internacional e formação interdisciplinar serão essenciais para ampliar a participação de universidades e centros de pesquisa no desenvolvimento dessas tecnologias.

Portanto, a computação quântica apresenta potencial para transformar determinados aspectos da inteligência artificial, especialmente nas áreas de aprendizado de máquina, otimização e simulação científica. Contudo, seus benefícios não devem ser interpretados como uma superioridade automática sobre a computação tradicional. A concretização desse potencial depende de avanços em hardware, correção de erros, desenvolvimento de algoritmos e demonstração experimental de vantagens práticas. Assim, o cenário mais plausível é o desenvolvimento progressivo de sistemas híbridos, nos quais recursos clássicos e quânticos sejam utilizados de forma complementar. A integração responsável dessas tecnologias poderá ampliar as fronteiras do conhecimento científico e oferecer novas ferramentas para enfrentar problemas complexos, desde que acompanhada de avaliações rigorosas, investimentos em pesquisa e compromisso com a democratização dos benefícios tecnológicos.
    """

In [11]:
#Avaliação da redação de exemplo
#Essa célula demonstra como usar o sistema de avaliação na redação de exemplo e exibir os resultados.
#mudou de 0-1 pra 0-10

In [12]:
# Avalia a redação de exemplo

result = grade_essay(sample_essay)

# Converte as pontuações de 0-1 para 0-10
final_score = result['final_score'] * 10
relevance_score = result['relevance_score'] * 10
grammar_score = result['grammar_score'] * 10
structure_score = result['structure_score'] * 10
depth_score = result['depth_score'] * 10

# Exibe os resultados
print(f"Pontuação Final da Redação: {final_score:.2f}/10\n")
print(f"Pontuação de Relevância: {relevance_score:.2f}/10")
print(f"Pontuação de Gramática: {grammar_score:.2f}/10")
print(f"Pontuação de Estrutura: {structure_score:.2f}/10")
print(f"Pontuação de Profundidade: {depth_score:.2f}/10")

Pontuação Final da Redação: 9.32/10

Pontuação de Relevância: 9.60/10
Pontuação de Gramática: 9.70/10
Pontuação de Estrutura: 9.60/10
Pontuação de Profundidade: 8.60/10
